In [ ]:
!!pip install faiss-cpu
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import pickle
import os
import glob


df = pd.read_csv("hf://datasets/nbertagnolli/counsel-chat/20220401_counsel_chat.csv")
print(df)

      questionID                                 questionTitle  \
0              0     Do I have too many issues for counseling?   
1              0     Do I have too many issues for counseling?   
2              0     Do I have too many issues for counseling?   
3              0     Do I have too many issues for counseling?   
4              0     Do I have too many issues for counseling?   
...          ...                                           ...   
2770         939  Are some clients more difficult than others?   
2771         939  Are some clients more difficult than others?   
2772         939  Are some clients more difficult than others?   
2773         939  Are some clients more difficult than others?   
2774         939  Are some clients more difficult than others?   

                                           questionText  \
0     I have so many issues to address. I have a his...   
1     I have so many issues to address. I have a his...   
2     I have so many issues to

In [ ]:
print(df.shape)           # How many rows/columns
print(df.columns.tolist()) # Column names
print(df.head(3))          # First 3 rows

(2775, 10)
['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views']
   questionID                              questionTitle  \
0           0  Do I have too many issues for counseling?   
1           0  Do I have too many issues for counseling?   
2           0  Do I have too many issues for counseling?   

                                        questionText  \
0  I have so many issues to address. I have a his...   
1  I have so many issues to address. I have a his...   
2  I have so many issues to address. I have a his...   

                                        questionLink       topic  \
0  https://counselchat.com/questions/do-i-have-to...  depression   
1  https://counselchat.com/questions/do-i-have-to...  depression   
2  https://counselchat.com/questions/do-i-have-to...  depression   

                                       therapistInfo  \
0  Jennifer MolinariHypnotherapist & Licensed Cou...  

In [ ]:
print(df['topic'].value_counts())

topic
depression                  465
anxiety                     358
counseling-fundamentals     270
intimacy                    248
relationships               202
parenting                   191
family-conflict             144
trauma                      102
self-esteem                 100
relationship-dissolution     98
behavioral-change            69
lgbtq                        65
marriage                     52
anger-management             48
spirituality                 47
substance-abuse              45
professional-ethics          41
grief-and-loss               37
workplace-relationships      36
social-relationships         25
diagnosis                    25
domestic-violence            21
eating-disorders             16
legal-regulatory             14
addiction                    13
stress                       13
sleep-improvement            11
children-adolescents          8
human-sexuality               7
military-issues               3
self-harm                     1
Na

In [ ]:
print(df.isnull().sum())

questionID         0
questionTitle      0
questionText     139
questionLink       0
topic              0
therapistInfo      0
therapistURL       0
answerText        26
upvotes            0
views              0
dtype: int64


In [ ]:
df['answer_len'] = df['answerText'].apply(lambda x: len(str(x).split()))
print(df['answer_len'].describe())

count    2775.000000
mean      166.388829
std       118.998309
min         1.000000
25%        86.000000
50%       134.000000
75%       212.000000
max       939.000000
Name: answer_len, dtype: float64


In [ ]:
import pandas as pd
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
import os

df = pd.read_csv("hf://datasets/nbertagnolli/counsel-chat/20220401_counsel_chat.csv")

def clean_text(text):
    text = str(text)
    text = re.sub(r'<.*?>', '', text)      # Remove HTML tags
    text = re.sub(r'http\S+', '', text)   # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)  # Keep punctuation
    text = text.lower().strip()
    return text

df['clean_question'] = df['questionText'].apply(clean_text)
df['clean_answer'] = df['answerText'].apply(clean_text)

# Drop duplicates and short answers
df = df.drop_duplicates(subset=['clean_question'])
df = df[df['clean_answer'].apply(lambda x: len(x.split()) > 20)]

# Create the directory if it doesn't exist
os.makedirs('data', exist_ok=True)
df.to_csv("data/cleaned_counsel_chat.csv", index=False)
print(f"Clean dataset: {len(df)} rows")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Clean dataset: 857 rows


In [ ]:
df = pd.read_csv("data/cleaned_counsel_chat.csv")

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings for all Q&A pairs
print("Creating embeddings...")
# Ensure 'clean_question' and 'clean_answer' columns contain only strings, converting any NaN (float) to empty strings
questions = df['clean_question'].fillna('').tolist()
answers   = df['clean_answer'].fillna('').tolist()

q_embeddings = embedder.encode(questions, show_progress_bar=True)

# Build FAISS index
dimension = q_embeddings.shape[1]  # 384 for MiniLM
index = faiss.IndexFlatL2(dimension)
index.add(np.array(q_embeddings, dtype='float32'))

# Create the models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save index and answers
faiss.write_index(index, "models/faiss_index.bin")
with open("models/answers.pkl", "wb") as f:
    pickle.dump(answers, f)

print(f"Vector store built: {index.ntotal} vectors")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Vector store built: 857 vectors


To get a Hugging Face token, go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and generate a new token. Once you have it, add it to Colab's secrets manager named `HF_TOKEN` (under the '🔑' icon in the left panel). Then, you can load it into your environment:

In [ ]:
from google.colab import userdata
import os

# Load the token from Colab secrets. Ensure the secret is named 'HF_TOKEN' in Colab's secrets manager.
hf_token = userdata.get('HF_TOKEN')

# Set the token as an environment variable
os.environ['HF_TOKEN'] = hf_token

print("Hugging Face token loaded and set as environment variable.")

Hugging Face token loaded and set as environment variable.


After setting the token, you might want to re-run the cell where `SentenceTransformer` was loaded to see if the warning disappears and if there's any performance improvement.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, pickle, numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("models/faiss_index.bin")
with open("models/answers.pkl", "rb") as f:
    answers = pickle.load(f)

def retrieve(query, top_k=3):
    query_vec = embedder.encode([query])
    distances, indices = index.search(
        np.array(query_vec, dtype='float32'), top_k
    )
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "answer": answers[idx],
            "score": float(distances[0][i])
        })
    return results
    # Test it
query = "I feel very anxious about my future"
results = retrieve(query)
for r in results:
    print("Score:", round(r['score'], 3))
    print("Answer:", r['answer'][:200])
    print("---")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Score: 0.8
Answer: this is a very common question in my practice. panic attacks typically emerge from an underlying issue ex. depression, low selfesteem, fears. to decrease your anxiety symptoms its recommended to seek 
---
Score: 0.967
Answer: tell your partner about this so that ideally the person has a chance to be supportive and reassuring, as well as listen to your specific worries and fears.initiate this type of discussion at a time wh
---
Score: 0.977
Answer: a good start is to pay attention to some basic issues sleep, nutrition, exercise and socially supportive relationships. a great car on an empty tank will not get you very far.
---


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def generate_response(user_input, chat_history_ids=None):
    new_input = tokenizer.encode(
        user_input + tokenizer.eos_token, return_tensors='pt'
    )
    bot_input = (
        torch.cat([chat_history_ids, new_input], dim=-1)
        if chat_history_ids is not None else new_input
    )
    output = model.generate(
        bot_input,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=100,
        top_p=0.7,
        temperature=0.8
    )
    response = tokenizer.decode(
        output[:, bot_input.shape[-1]:][0],
        skip_special_tokens=True
   )
    return response, output

resp, _ = generate_response("I feel really sad today")
print(resp)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


It's ok , I feel the same way .


In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import os

# Check if the file exists before trying to read it
file_path = "data/cleaned_counsel_chat.csv"
if not os.path.exists(file_path):
    # If the file doesn't exist, raise a more informative error
    raise FileNotFoundError(
        f"The file '{file_path}' was not found. "
        "Please ensure that the data cleaning and preprocessing cell "
        "(the one that generates 'cleaned_counsel_chat.csv') has been executed successfully."
    )

df = pd.read_csv(file_path)
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
tokenizer.pad_token = tokenizer.eos_token # Add this line to define the padding token

# Ensure 'clean_question' and 'clean_answer' are string types to prevent TypeError with NaN values
df['clean_question'] = df['clean_question'].astype(str)
df['clean_answer'] = df['clean_answer'].astype(str)

# Format as conversational pairs
def format_conversation(row):
    return {
        "input_ids": tokenizer.encode(
            row['clean_question'] + tokenizer.eos_token +
            row['clean_answer'] + tokenizer.eos_token,
            truncation=True, max_length=512, padding='max_length'
        )
    }

formatted = df.apply(format_conversation, axis=1).tolist()

# Split train/val
split = int(0.9 * len(formatted))
train_data = formatted[:split]
val_data = formatted[split:]

import json
# Ensure the 'data' directory exists before writing JSON files
os.makedirs('data', exist_ok=True)
with open("data/train.json","w") as f: json.dump(train_data,f)
with open("data/val.json","w") as f:   json.dump(val_data,f)

print(f"Train: {len(train_data)}, Val: {len(val_data)}")

Train: 771, Val: 86


In [ ]:
from google.colab import files
uploaded = files.upload()

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

/content/data/cleaned_counsel_chat.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv


In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    print(root)

/content
/content/.config
/content/.config/logs
/content/.config/logs/2026.06.04
/content/.config/configurations
/content/data
/content/models
/content/sample_data


In [ ]:
import glob

glob.glob('/content/**/*.csv', recursive=True)

['/content/sample_data/california_housing_train.csv',
 '/content/sample_data/california_housing_test.csv',
 '/content/sample_data/mnist_train_small.csv',
 '/content/sample_data/mnist_test.csv']

In [ ]:
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from torch.utils.data import Dataset
import torch, json
import os
from google.colab import userdata
import pandas as pd # Added import for data processing
import re # Added import for text cleaning
import nltk # Added import for nltk.download

# Ensure NLTK stopwords are downloaded for potential use in text cleaning
try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    print("Downloading NLTK stopwords...")
    nltk.download('stopwords')

# Load the HF_TOKEN from Colab secrets to authenticate with Hugging Face Hub
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Hugging Face token loaded and set as environment variable.")
except Exception as e:
    print(f"Could not load HF_TOKEN from Colab secrets: {e}")
    print("Proceeding without Hugging Face token. You may encounter rate limits or slower downloads.")

# Initialize tokenizer early, as it's needed for data preparation
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
tokenizer.pad_token = tokenizer.eos_token

# Function to prepare data files if they are missing
def prepare_data_files():
    print("Preparing data files (train.json, val.json) as they were not found...")
    cleaned_csv_path = "data/cleaned_counsel_chat.csv"

    # If cleaned_counsel_chat.csv is not found, generate it
    if not os.path.exists(cleaned_csv_path):
        print(f"'{cleaned_csv_path}' not found. Generating it now...")

        original_df = pd.read_csv("hf://datasets/nbertagnolli/counsel-chat/20220401_counsel_chat.csv")

        def clean_text_local(text):
            text = str(text)
            text = re.sub(r'<.*?>', '', text)      # Remove HTML tags
            text = re.sub(r'http\\S+', '', text)   # Remove URLs
            text = re.sub(r'[^a-zA-Z0-9\\s.,!?]', '', text)  # Keep punctuation
            text = text.lower().strip()
            return text

        original_df['clean_question'] = original_df['questionText'].apply(clean_text_local)
        original_df['clean_answer'] = original_df['answerText'].apply(clean_text_local)

        # Drop duplicates and short answers
        original_df = original_df.drop_duplicates(subset=['clean_question'])
        original_df = original_df[original_df['clean_answer'].apply(lambda x: len(x.split()) > 20)]

        os.makedirs('data', exist_ok=True)
        original_df.to_csv(cleaned_csv_path, index=False)
        print(f"'{cleaned_csv_path}' generated with {len(original_df)} rows.")

    # Now that cleaned_csv_path is guaranteed to exist, proceed with train/val.json
    df = pd.read_csv(cleaned_csv_path) # Use the now guaranteed existing file

    # Ensure 'clean_question' and 'clean_answer' are string types
    df['clean_question'] = df['clean_question'].astype(str)
    df['clean_answer'] = df['clean_answer'].astype(str)

    def format_conversation(row):
        return {
            "input_ids": tokenizer.encode( # Use the global tokenizer
                row['clean_question'] + tokenizer.eos_token +
                row['clean_answer'] + tokenizer.eos_token,
                truncation=True, max_length=512, padding='max_length'
            )
        }

    formatted = df.apply(format_conversation, axis=1).tolist()
    split = int(0.9 * len(formatted))
    train_data = formatted[:split]
    val_data = formatted[split:]

    os.makedirs('data', exist_ok=True)
    with open("data/train.json","w") as f: json.dump(train_data,f)
    with open("data/val.json","w") as f:   json.dump(val_data,f)
    print(f"Data files prepared: Train: {len(train_data)}, Val: {len(val_data)}")


class CounselDataset(Dataset):
    def __init__(self, path):
        with open(path) as f: self.data = json.load(f)
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        ids = self.data[i]["input_ids"]
        # Return lists of integers, let DataCollator handle tensor conversion and padding
        return {"input_ids": ids,
                "labels": ids}

model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

# Check and prepare data if not found
if not os.path.exists("data/train.json") or not os.path.exists("data/val.json"):
    prepare_data_files()

train_ds = CounselDataset("data/train.json")
val_ds   = CounselDataset("data/val.json")
args = TrainingArguments(
    output_dir="models/finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=200,
    # evaluation_strategy="epoch", # This argument is not recognized in older transformers versions
    # save_strategy="epoch",       # This argument is not recognized in older transformers versions
    # load_best_model_at_end=True, # This argument is not recognized in older transformers versions
    fp16=False, # Changed from fp16=torch.cuda.is_available() to disable FP16 training
    logging_steps=50
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
trainer.train()
model.save_pretrained("models/finetuned_final")

Hugging Face token loaded and set as environment variable.


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Step,Training Loss
50,3.871957
100,0.000000
150,0.000000
200,0.000000
250,0.000000
300,0.000000
350,0.000000
400,0.000000
450,0.000000
500,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import faiss, pickle, numpy as np, torch

embedder  = SentenceTransformer('all-MiniLM-L6-v2')
index     = faiss.read_index("models/faiss_index.bin")
with open("models/answers.pkl","rb") as f: answers = pickle.load(f)

tokenizer = AutoTokenizer.from_pretrained("models/finetuned_final")
model     = AutoModelForCausalLM.from_pretrained("models/finetuned_final")
tokenizer.pad_token = tokenizer.eos_token

def rag_respond(user_input, top_k=2):
    # Step 1: Retrieve
    vec = embedder.encode([user_input])
    _, idxs = index.search(np.array(vec, dtype='float32'), top_k)
    context = " ".join([answers[i][:300] for i in idxs[0]])

    # Step 2: Build prompt with context
    prompt = (
        f"Context from therapist: {context}\n\n"
        f"User says: {user_input}\n"
        f"Empathetic response:"
    )
    # Step 3: Generate
    inputs = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt')
    output = model.generate(
        inputs, max_new_tokens=150,
        do_sample=False, # Changed to False to disable sampling and prevent RuntimeError
        top_p=0.85, temperature=0.75,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

print(rag_respond("I feel so alone and nobody understands me"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
import re

CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "don't want to live",
    "hurt myself", "self harm", "want to die", "no reason to live",
    "overdose", "cut myself", "jump off"
]

CRISIS_RESPONSE = """
I hear that you're going through something very painful right now.
Please reach out to a professional who can help immediately:

iCall (India): 9152987821
Vandrevala Foundation: 1860-2662-345 (24/7)
iCall email: icall@tiss.edu

You are not alone. Please call one of these numbers right now.
"""
def safety_check(text):
    text_lower = text.lower()
    for kw in CRISIS_KEYWORDS:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return True, CRISIS_RESPONSE
    return False, None

# Integrate into main pipeline
def safe_rag_respond(user_input):
    is_crisis, crisis_msg = safety_check(user_input)
    if is_crisis:
        return crisis_msg, "CRISIS_DETECTED"
    response = rag_respond(user_input)
    return response, "NORMAL"

text, flag = safe_rag_respond("I want to hurt myself")
print(flag, "\n", text)

CRISIS_DETECTED 
 
I hear that you're going through something very painful right now.
Please reach out to a professional who can help immediately:

iCall (India): 9152987821
Vandrevala Foundation: 1860-2662-345 (24/7)
iCall email: icall@tiss.edu

You are not alone. Please call one of these numbers right now.



In [ ]:
from collections import deque
import re # Needed for safety_check
from sentence_transformers import SentenceTransformer # Needed for rag_respond
from transformers import AutoModelForCausalLM, AutoTokenizer # Needed for rag_respond
import faiss, pickle, numpy as np, torch # Needed for rag_respond

# Definition of safety_check (copied from cell gLPUrIW7Vl4k to resolve NameError)
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "don't want to live",
    "hurt myself", "self harm", "want to die", "no reason to live",
    "overdose", "cut myself", "jump off"
]

CRISIS_RESPONSE = """
I hear that you're going through something very painful right now.
Please reach out to a professional who can help immediately:

iCall (India): 9152987821
Vandrevala Foundation: 1860-2662-345 (24/7)
iCall email: icall@tiss.edu

You are not alone. Please call one of these numbers right now.
"""
def safety_check(text):
    text_lower = text.lower()
    for kw in CRISIS_KEYWORDS:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return True, CRISIS_RESPONSE
    return False, None

# Definition of rag_respond (copied from cell BeATLELZJLXY to resolve NameError)
# Initializing models and data for RAG. This part is typically in a setup cell,
# but is included here to make this cell self-contained and resolve the NameError.
embedder  = SentenceTransformer('all-MiniLM-L6-v2')
index     = faiss.read_index("models/faiss_index.bin")
with open("models/answers.pkl","rb") as f: answers = pickle.load(f)

tokenizer = AutoTokenizer.from_pretrained("models/finetuned_final")
model     = AutoModelForCausalLM.from_pretrained("models/finetuned_final")
tokenizer.pad_token = tokenizer.eos_token

def rag_respond(user_input, top_k=2):
    # Step 1: Retrieve
    vec = embedder.encode([user_input])
    _, idxs = index.search(np.array(vec, dtype='float32'), top_k)
    context = " ".join([answers[i][:300] for i in idxs[0]])

    # Step 2: Build prompt with context
    prompt = (
        f"Context from therapist: {context}\n\n"
        f"User says: {user_input}\n"
        f"Empathetic response:"
    )
    # Step 3: Generate
    inputs = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors='pt')
    output = model.generate(
        inputs, max_new_tokens=150,
        do_sample=False, # Changed to False to disable sampling and prevent RuntimeError
        top_p=0.85, temperature=0.75,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

class ConversationMemory:
    def __init__(self, max_turns=5):
        self.history = deque(maxlen=max_turns)

    def add(self, role, content):
        self.history.append({"role": role, "content": content})

    def get_context(self):
        context_parts = []
        for turn in self.history:
            prefix = "User" if turn["role"] == "user" else "Bot"
            context_parts.append(f"{prefix}: {turn['content']}")
        return "\n".join(context_parts)

    def clear(self):
        self.history.clear()

# Updated pipeline with memory
memory = ConversationMemory(max_turns=5)
def chat_with_memory(user_input):
    is_crisis, crisis_msg = safety_check(user_input)
    if is_crisis:
        memory.clear()
        return crisis_msg

    history_ctx = memory.get_context()
    full_input = f"{history_ctx}\nUser: {user_input}" if history_ctx else user_input

    response = rag_respond(full_input)

    memory.add("user", user_input)
    memory.add("bot", response)
    return response

# Test multi-turn
print(chat_with_memory("I feel anxious lately"))
print(chat_with_memory("It gets worse at night"))  # Should remember context


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

In [ ]:
import re
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import faiss, pickle, numpy as np, torch

# --- Safety Check Definitions ---
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "don't want to live",
    "hurt myself", "self harm", "want to die", "no reason to live",
    "overdose", "cut myself", "jump off"
]

CRISIS_RESPONSE = """
I hear that you're going through something very painful right now.
Please reach out to a professional who can help immediately:

iCall (India): 9152987821
Vandrevala Foundation: 1860-2662-345 (24/7)
iCall email: icall@tiss.edu

You are not alone. Please call one of these numbers right now.
"""
def safety_check(text):
    text_lower = text.lower()
    for kw in CRISIS_KEYWORDS:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return True, CRISIS_RESPONSE
    return False, None

# --- RAG Setup and Function Definition ---
# Ensure models and indices are loaded
embedder  = SentenceTransformer('all-MiniLM-L6-v2')
index     = faiss.read_index("models/faiss_index.bin")
with open("models/answers.pkl","rb") as f: answers = pickle.load(f)

tokenizer_rag = AutoTokenizer.from_pretrained("models/finetuned_final") # Renamed to avoid potential conflict
model_rag     = AutoModelForCausalLM.from_pretrained("models/finetuned_final") # Renamed
tokenizer_rag.pad_token = tokenizer_rag.eos_token

def rag_respond(user_input, top_k=2):
    # Step 1: Retrieve
    vec = embedder.encode([user_input])
    _, idxs = index.search(
        np.array(vec, dtype='float32'), top_k
    )
    context = " ".join([answers[i][:300] for i in idxs[0]])

    # Step 2: Build prompt with context
    prompt = (
        f"Context from therapist: {context}\n\n"
        f"User says: {user_input}\n"
        f"Empathetic response:"
    )
    # Step 3: Generate
    inputs = tokenizer_rag.encode(prompt + tokenizer_rag.eos_token, return_tensors='pt')
    output = model_rag.generate(
        inputs, max_new_tokens=150,
        do_sample=False, top_p=0.85, temperature=0.75, # Changed do_sample to False
        pad_token_id=tokenizer_rag.eos_token_id
    )
    response = tokenizer_rag.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

# --- Emotion Classifier Setup ---
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)

def detect_emotion(text):
    results = emotion_classifier(text)
    if len(results) > 0 and isinstance(results[0], list): # Handle case where it's a list of lists
        results = results[0]

    results.sort(key=lambda x: x['score'], reverse=True)
    top = results[0]
    return top['label'], round(top['score'], 3)

def adaptive_respond(user_input):
    is_crisis, crisis_msg = safety_check(user_input)
    if is_crisis:
        return crisis_msg, "CRISIS_DETECTED" # Added second return value for consistency

    emotion, confidence = detect_emotion(user_input)

    # Adapt tone based on emotion
    if emotion in ["sadness", "fear"] and confidence > 0.6:
        prefix = "I hear how difficult this is. "
    elif emotion == "anger":
        prefix = "It sounds like you're really frustrated. "
    elif emotion == "joy":
        prefix = "That's wonderful to hear. "
    else:
        prefix = ""

    response = rag_respond(user_input)
    return prefix + response, emotion

resp, emo = adaptive_respond("I am terrified about my upcoming exam")
print(f"Detected emotion: {emo}")
print(f"Response: {resp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Detected emotion: fear
Response: I hear how difficult this is. 


In [ ]:
from collections import deque

class ConversationMemory:
    def __init__(self, max_turns=5):
        self.history = deque(maxlen=max_turns)

    def add(self, role, content):
        self.history.append({"role": role, "content": content})

    def get_context(self):
        context_parts = []
        for turn in self.history:
            prefix = "User" if turn["role"] == "user" else "Bot"
            context_parts.append(f"{prefix}: {turn['content']}")
        return "\n".join(context_parts)

    def clear(self):
        self.history.clear()
# Updated pipeline with memory
memory = ConversationMemory(max_turns=5)

def chat_with_memory(user_input):
    is_crisis, crisis_msg = safety_check(user_input)
    if is_crisis:
        memory.clear()
        return crisis_msg

    history_ctx = memory.get_context()
    full_input = f"{history_ctx}\nUser: {user_input}" if history_ctx else user_input

    response = rag_respond(full_input)

    memory.add("user", user_input)
    memory.add("bot", response)
    return response

# Test multi-turn
print(chat_with_memory("I feel anxious lately"))
print(chat_with_memory("It gets worse at night"))  # Should remember context

In [ ]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=True
)

def detect_emotion(text):
    results = emotion_classifier(text)
    # Ensure results is a list of dictionaries, handling potential list of lists output
    if len(results) > 0 and isinstance(results[0], list):
        results = results[0]

    results.sort(key=lambda x: x['score'], reverse=True)
    top = results[0]
    return top['label'], round(top['score'], 3)

def adaptive_respond(user_input):
    is_crisis, crisis_msg = safety_check(user_input)
    if is_crisis:
        return crisis_msg

    emotion, confidence = detect_emotion(user_input)
      # Adapt tone based on emotion
    if emotion in ["sadness", "fear"] and confidence > 0.6:
        prefix = "I hear how difficult this is. "
    elif emotion == "anger":
        prefix = "It sounds like you're really frustrated. "
    elif emotion == "joy":
        prefix = "That's wonderful to hear. "
    else:
        prefix = ""

    response = rag_respond(user_input)
    return prefix + response, emotion

resp, emo = adaptive_respond("I am terrified about my upcoming exam")
print(f"Detected emotion: {emo}")
print(f"Response: {resp}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Detected emotion: fear
Response: I hear how difficult this is. 


In [ ]:
from transformers import pipeline

topic_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

TOPICS = [
    "anxiety", "depression", "relationships",
    "self-esteem", "grief", "trauma",
    "workplace stress", "family issues", "sleep problems"
]

def classify_topic(text):
    result = topic_classifier(text, candidate_labels=TOPICS)
    top_topic = result['labels'][0]
    top_score = round(result['scores'][0], 3)
    return top_topic, top_score

topic, score = classify_topic("My boss keeps criticizing me and I dread going to work")
print(f"Topic: {topic} | Confidence: {score}")
# Use in retrieval — filter FAISS results by topic
def topic_aware_respond(user_input):
    topic, _ = classify_topic(user_input)
    response = rag_respond(user_input)
    return response, topic

In [37]:
# Install detoxify
!pip install detoxify

# IMPORTS
import numpy as np
import re
import faiss
import pickle
import torch

from detoxify import Detoxify
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline # Combine pipeline here

# --- SAFETY CHECK (CRISIS DETECTION) ---
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "don't want to live",
    "hurt myself", "self harm", "want to die", "no reason to live",
    "overdose", "cut myself", "jump off"
]
CRISIS_RESPONSE = (
    "I hear that you're going through something very painful right now.\n"
    "Please reach out to a professional who can help immediately:\n\n"
    "iCall (India): 9152987821\n"
    "Vandrevala Foundation: 1860-2662-345 (24/7)\n"
    "iCall email: icall@tiss.edu\n\n"
    "You are not alone. Please call one of these numbers right now."
)
def safety_check(text):
    text_lower = text.lower()
    for kw in CRISIS_KEYWORDS:
        if re.search(r'\\b' + re.escape(kw) + r'\\b', text_lower):
            return True, CRISIS_RESPONSE
    return False, None

# --- RETRIEVAL-AUGMENTED GENERATION (RAG) SETUP ---
embedder = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("models/faiss_index.bin")
with open("models/answers.pkl", "rb") as f:
    answers = pickle.load(f)

tokenizer_rag = AutoTokenizer.from_pretrained("models/finetuned_final")
model_rag = AutoModelForCausalLM.from_pretrained("models/finetuned_final")
tokenizer_rag.pad_token = tokenizer_rag.eos_token

def rag_respond(user_input, top_k=2):
    vec = embedder.encode([user_input])
    _, idxs = index.search(np.array(vec, dtype='float32'), top_k)
    context = " ".join([answers[i][:300] for i in idxs[0]])

    prompt = (
        f"Context from therapist: {context}\n\n"
        f"User says: {user_input}\n"
        f"Empathetic response:"
    )
    inputs = tokenizer_rag.encode(prompt + tokenizer_rag.eos_token, return_tensors='pt')
    output = model_rag.generate(
        inputs, max_new_tokens=150,
        do_sample=False,
        top_p=0.85, temperature=0.75,
        pad_token_id=tokenizer_rag.eos_token_id
    )
    response = tokenizer_rag.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

# --- TOPIC CLASSIFICATION ---
topic_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
TOPICS = [
    "anxiety", "depression", "relationships",
    "self-esteem", "grief", "trauma",
    "workplace stress", "family issues", "sleep problems"
]
def classify_topic(text):
    result = topic_classifier(text, candidate_labels=TOPICS)
    top_topic = result['labels'][0]
    top_score = round(result['scores'][0], 3)
    return top_topic, top_score

def topic_aware_respond(user_input):
    topic, _ = classify_topic(user_input)
    response = rag_respond(user_input)
    return response, topic

# --- TOXICITY DETECTION ---
toxic_model = Detoxify('original')
TOXICITY_THRESHOLD = 0.5

def is_toxic(text):
    scores = toxic_model.predict(text)
    max_score = max(scores.values())
    return max_score > TOXICITY_THRESHOLD, scores

FALLBACK_RESPONSE = (
    "I want to make sure I'm giving you the support you deserve. "
    "Could you tell me more about how you're feeling right now?"
)

# --- MAIN GENERATION FUNCTION ---
def safe_generate(user_input):
    is_crisis, crisis_msg = safety_check(user_input)
    if is_crisis: return crisis_msg

    response, topic = topic_aware_respond(user_input)

    toxic, scores = is_toxic(response)
    if toxic:
        print(f"Toxicity detected: {scores}")
        return FALLBACK_RESPONSE

    return response

# --- TEST ---
print(safe_generate("I feel completely worthless"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [38]:
import json, datetime, uuid

class SessionLogger:
    def __init__(self, log_file="logs/sessions.jsonl"):
        self.log_file = log_file
        self.session_id = str(uuid.uuid4())[:8]

    def log_turn(self, user_input, response, emotion, topic, latency_ms):
        entry = {
            "session_id": self.session_id,
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "user_input": user_input[:200],
            "response_snippet": response[:200],
            "emotion": emotion,
            "topic": topic,
            "latency_ms": round(latency_ms)
        }
        with open(self.log_file, "a") as f:
            f.write(json.dumps(entry) + "\n")
   # Integrate into pipeline
import time
logger = SessionLogger()

def full_pipeline(user_input):
    start = time.time()
    is_crisis, crisis_msg = safety_check(user_input)
    if is_crisis:
        logger.log_turn(user_input, crisis_msg, "crisis", "crisis", 0)
        return crisis_msg

    emotion, _ = detect_emotion(user_input)
    response, topic = topic_aware_respond(user_input)
    toxic, _ = is_toxic(response)
    if toxic: response = FALLBACK_RESPONSE

    latency = (time.time() - start) * 1000
    logger.log_turn(user_input, response, emotion, topic, latency)
    return response

In [44]:
!pip install streamlit
import streamlit as st

st.set_page_config(
    page_title="MindBridge — Mental Health Support",
    page_icon="heart",
    layout="centered"
)

st.markdown("""
<style>
.crisis-box{background:#fff3cd;border:1px solid #ffc107;border-radius:8px;padding:12px;margin:8px 0}
.chat-msg{padding:8px 12px;border-radius:8px;margin:4px 0;max-width:80%}
</style>
""", unsafe_allow_html=True)
st.title("MindBridge")
st.caption("A safe space to talk. Not a substitute for professional help.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("How are you feeling today?")

if user_input:
    st.session_state.messages.append({"role":"user","content":user_input})
    with st.chat_message("user"):
        st.write(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = full_pipeline(user_input)
        st.write(response)

    st.session_state.messages.append({"role":"assistant","content":response})

st.sidebar.markdown("**Emergency contacts**")
st.sidebar.markdown("iCall: 9152987821")
st.sidebar.markdown("Vandrevala: 1860-2662-345")

2026-06-11 13:24:59.515 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 13:24:59.516 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 13:24:59.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 13:24:59.519 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 13:24:59.520 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 13:24:59.522 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 13:24:59.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-11 13:24:59.524 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator(_root_container=1, _parent=DeltaGenerator())

In [42]:
import streamlit as st

EMOTION_COLORS = {
    "sadness": ("#EBF4FB", "Feeling sad"),
    "fear":    ("#FEF3E2", "Feeling anxious"),
    "anger":   ("#FDEDEC", "Feeling frustrated"),
    "joy":     ("#EAFAF1", "Feeling positive"),
    "neutral": ("#F2F3F4", "Neutral")
}

def show_emotion_badge(emotion):
    color, label = EMOTION_COLORS.get(emotion, ("#F2F3F4", emotion))
    st.markdown(
        f'<div style="background:{color};border-radius:20px;'
        f'padding:4px 12px;display:inline-block;font-size:13px;'
        f'margin-bottom:8px">{label}</div>',
        unsafe_allow_html=True
    )

if user_input:
    emotion, _ = detect_emotion(user_input)
    show_emotion_badge(emotion)
    # ... rest of response generation
# Also add conversation stats in sidebar
if st.session_state.messages:
    total = len([m for m in st.session_state.messages if m["role"]=="user"])
    st.sidebar.metric("Messages sent", total)


2026-06-11 13:23:38.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [49]:
import pytest
import re
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from detoxify import Detoxify

# --- Copying necessary function definitions to make tests self-contained ---

# safety_check function (from cell r8mhq7JbyeMD/gLPUrIW7Vl4k)
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "don't want to live",
    "hurt myself", "self harm", "want to die", "no reason to live",
    "overdose", "cut myself", "jump off"
]
CRISIS_RESPONSE = (
    "I hear that you're going through something very painful right now.\n"
    "Please reach out to a professional who can help immediately:\n\n"
    "iCall (India): 9152987821\n"
    "Vandrevala Foundation: 1860-2662-345 (24/7)\n"
    "iCall email: icall@tiss.edu\n\n"
    "You are not alone. Please call one of these numbers right now."
)
def safety_check(text):
    text_lower = text.lower()
    for kw in CRISIS_KEYWORDS:
        if re.search(r'\\b' + re.escape(kw) + r'\\b', text_lower):
            return True, CRISIS_RESPONSE
    return False, None

# is_toxic function (from cell r8mhq7JbyeMD)
toxic_model = Detoxify('original')
TOXICITY_THRESHOLD = 0.5
def is_toxic(text):
    scores = toxic_model.predict(text)
    max_score = max(scores.values())
    return max_score > TOXICITY_THRESHOLD, scores

# retrieve function (from cell 7Fq9pPSC2r6u)
# Load embedding model and FAISS index for retrieve function
embedder = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("models/faiss_index.bin")
with open("models/answers.pkl", "rb") as f:
    answers = pickle.load(f)
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query])
    distances, indices = index.search(
        np.array(query_vec, dtype='float32'), top_k
    )
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "answer": answers[idx],
            "score": float(distances[0][i])
        })
    return results

# --- Test Classes ---
class TestSafetyFilter:
    def test_crisis_keyword_detected(self):
        flagged, msg = safety_check("I want to kill myself")
        assert flagged == True
        assert "1860" in msg or "9152987821" in msg

    def test_normal_text_passes(self):
        flagged, _ = safety_check("I feel a bit anxious today")
        assert flagged == False

    def test_case_insensitive(self):
        flagged, _ = safety_check("I WANT TO END MY LIFE")
        assert flagged == True

class TestRetrieval:
    def test_returns_correct_count(self):
        results = retrieve("I feel depressed", top_k=3)
        assert len(results) == 3

    def test_results_have_score(self):
        results = retrieve("anxiety attack", top_k=2)
        for r in results:
            assert "score" in r
            assert "answer" in r

class TestToxicity:
    def test_toxic_text_flagged(self):
        toxic, _ = is_toxic("You are worthless and should disappear")
        assert toxic == True

    def test_safe_text_passes(self):
        toxic, _ = is_toxic("I am feeling better after talking to someone")
        assert toxic == False

# Run with: pytest tests/ -v

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [52]:
# Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.address=0.0.0.0"]

# requirements.txt (key packages)
# torch==2.0.1
# transformers==4.35.0
# sentence-transformers==2.2.2
# langchain==0.0.350
# faiss-cpu==1.7.4
# streamlit==1.28.0
# detoxify==0.5.1
# pandas==2.1.0

SyntaxError: invalid syntax (2788243575.py, line 2)

In [53]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.address=0.0.0.0"]


Overwriting Dockerfile


You also need to create the `requirements.txt` file which is referenced in the Dockerfile. Here's how you can do that:

In [54]:
%%writefile requirements.txt
torch==2.0.1
transformers==4.35.0
sentence-transformers==2.2.2
langchain==0.0.350
faiss-cpu==1.7.4
streamlit==1.28.0
detoxify==0.5.1
pandas==2.1.0

Overwriting requirements.txt


In [58]:
%%writefile test_chatbot.py
import pytest
import re
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from detoxify import Detoxify

# --- Copying necessary function definitions to make tests self-contained ---

# safety_check function
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "don't want to live",
    "hurt myself", "self harm", "want to die", "no reason to live",
    "overdose", "cut myself", "jump off"
]
CRISIS_RESPONSE = (
    "I hear that you're going through something very painful right now.\n"
    "Please reach out to a professional who can help immediately:\n\n"
    "iCall (India): 9152987821\n"
    "Vandrevala Foundation: 1860-2662-345 (24/7)\n"
    "iCall email: icall@tiss.edu\n\n"
    "You are not alone. Please call one of these numbers right now."
)
def safety_check(text):
    text_lower = text.lower()
    for kw in CRISIS_KEYWORDS:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return True, CRISIS_RESPONSE
    return False, None

# is_toxic function
toxic_model = Detoxify('original')
TOXICITY_THRESHOLD = 0.5
def is_toxic(text):
    scores = toxic_model.predict(text)
    max_score = max(scores.values())
    return max_score > TOXICITY_THRESHOLD, scores

# retrieve function
# Load embedding model and FAISS index for retrieve function
# Ensure these files are available in the current environment
embedder = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("models/faiss_index.bin")
with open("models/answers.pkl", "rb") as f:
    answers = pickle.load(f)
def retrieve(query, top_k=3):
    query_vec = embedder.encode([query])
    distances, indices = index.search(
        np.array(query_vec, dtype='float32'), top_k
    )
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "answer": answers[idx],
            "score": float(distances[0][i])
        })
    return results

# --- Test Classes ---
class TestSafetyFilter:
    def test_crisis_keyword_detected(self):
        flagged, msg = safety_check("I want to kill myself")
        assert flagged == True
        assert "1860" in msg or "9152987821" in msg

    def test_normal_text_passes(self):
        flagged, _ = safety_check("I feel a bit anxious today")
        assert flagged == False

    def test_case_insensitive(self):
        flagged, _ = safety_check("I WANT TO END MY LIFE")
        assert flagged == True

class TestRetrieval:
    def test_returns_correct_count(self):
        results = retrieve("I feel depressed", top_k=3)
        assert len(results) == 3

    def test_results_have_score(self):
        results = retrieve("anxiety attack", top_k=2)
        for r in results:
            assert "score" in r
            assert "answer" in r

class TestToxicity:
    def test_toxic_text_flagged(self):
        toxic, _ = is_toxic("You are worthless and should disappear")
        assert toxic == True

    def test_safe_text_passes(self):
        toxic, _ = is_toxic("I am feeling better after talking to someone")
        assert toxic == False


Overwriting test_chatbot.py


In [59]:
!pytest test_chatbot.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.8.9, typeguard-4.5.2, anyio-4.13.0
collected 7 items                                                              

test_chatbot.py .......                                                  [100%]

============================== 7 passed in 18.00s ==============================


In [60]:
!pytest test_chatbot.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.8.9, typeguard-4.5.2, anyio-4.13.0
collected 7 items                                                              

test_chatbot.py .......                                                  [100%]

============================== 7 passed in 16.91s ==============================


In [62]:
!git clone https://github.com/Ghodesaloni/mental-health-chatbot.git

Cloning into 'mental-health-chatbot'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 51.67 KiB | 2.72 MiB/s, done.
